In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import gseapy as gp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.colorbar as colorbar

from typing import Tuple, Literal
from anndata import AnnData
from itertools import product

from scipy.stats import pearsonr
import textwrap

#### dataset publication: https://www.nature.com/articles/s41467-019-14118-w

In [ ]:
for mod in [np, pd, sc, gp]:
    print(mod, mod.__version__)

In [ ]:
def vprint(msg, verbose):
    if verbose:
        print(msg)

def get_genesets(
    adata: AnnData,
    df_hcd_all: pd.DataFrame,
    celltype_signature: str,
    direction: Literal["upregulated", "downregulated", "both"],
    threshold_pval: float,
    threshold_lfc: float, 
) -> Tuple[dict[str, list[str]], pd.DataFrame]:

    # construct signature gene set
    select = ((df_hcd_all.adj_p_value <= threshold_pval) & (df_hcd_all.celltype == celltype_signature))
    if direction == "upregulated":
        select = select & (df_hcd_all.log_fc >= threshold_lfc)
    elif direction == "downregulated":
        select = select & (df_hcd_all.log_fc <= threshold_lfc)
    elif direction == "both":
        select = select & (df_hcd_all.log_fc.abs() >= threshold_lfc)         
    else:
        raise ValueError(f"Invalid direction: {direction}.")
    
    df_hcd_ct = df_hcd_all.loc[select]
    
    gene_set_dict = {}
    gene_set_df = pd.DataFrame()
    for cytokine_i, cytokine in enumerate(df_hcd_ct.cytokine.unique()):
        gene_set = df_hcd_ct.loc[df_hcd_ct.cytokine == cytokine].gene.values
        gene_set_shared = np.intersect1d(gene_set, adata.var_names)
        gene_set_df.loc[cytokine_i, "cytokine"] = cytokine
        gene_set_df.loc[cytokine_i, "num_genes_signature"] = len(gene_set)
        gene_set_df.loc[cytokine_i, "num_shared_genes_signature"] = len(gene_set_shared)
        gene_set_df.loc[cytokine_i, "frac_shared_genes_signature"] = len(gene_set_shared) / len(gene_set)
        gene_set_dict[cytokine] = gene_set_shared        

    return gene_set_dict, gene_set_df


def compute_mu_and_sigma(adata: AnnData, contrast_column: str, condition: str) -> pd.DataFrame:

    group = adata[adata.obs[contrast_column] == condition]
    num_cells = group.shape[0]
    X = group.X.toarray() if hasattr(group.X, "toarray") else group.X
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0, ddof=1)
    return {"mu": mu, "sigma": sigma, "num_cells": num_cells}


def compute_s2n(
    adata: AnnData, 
    contrast_column: str, 
    condition_1: str, 
    condition_2: str,
    precomputed_stats: dict | None = None
    
) -> (pd.DataFrame, pd.DataFrame):
    """
    Compute the signal-to-noise ratio (S2N) for each gene between two conditions in an AnnData object.

    Parameters:
    - adata: AnnData object with gene expression data.
    - contrast_column: Key in `adata.obs` indicating the condition labels (e.g. "disease_state").
    - condition_1: Name of the first condition (e.g., "flare").
    - condition_2: Name of the second condition (e.g., "healthy").
    
    Returns:
    - s2n_scores: pandas Series of S2N values indexed by gene names.
    """

    if precomputed_stats is None:
    
        # Select cells for each condition
        group1 = adata[adata.obs[contrast_column] == condition_1]
        group2 = adata[adata.obs[contrast_column] == condition_2]
        
        # number of cells per condition
        num_cells_1 = group1.shape[0]
        num_cells_2 = group2.shape[0]
        
        # Get expression matrices
        X1 = group1.X.toarray() if hasattr(group1.X, "toarray") else group1.X
        X2 = group2.X.toarray() if hasattr(group2.X, "toarray") else group2.X
    
        # Compute mean and std per gene
        mu1 = np.mean(X1, axis=0)
        mu2 = np.mean(X2, axis=0)
        sigma1 = np.std(X1, axis=0, ddof=1)
        sigma2 = np.std(X2, axis=0, ddof=1)

    else:
        vprint("Using precomputed stats", True)
        num_cells_1 = precomputed_stats[condition_1]["num_cells"]
        num_cells_2 = precomputed_stats[condition_2]["num_cells"]
        mu1 = precomputed_stats[condition_1]["mu"]
        mu2 = precomputed_stats[condition_2]["mu"]
        sigma1 = precomputed_stats[condition_1]["sigma"]
        sigma2 = precomputed_stats[condition_2]["sigma"]

    # Compute S2N
    s2n = (mu1 - mu2) / (sigma1 + sigma2 + 1e-8)  # epsilon to avoid division by zero

    num_cells = pd.DataFrame(index=[f"{condition_1}_vs_{condition_2}"], columns=["num_cells_1", "num_cells_2"], data=[[num_cells_1, num_cells_2]])
    stats = pd.DataFrame(s2n, index=adata.var_names, columns=[f"{condition_1}_vs_{condition_2}"])
    
    return stats, num_cells

  
def compute_ranking_statistic(adata: AnnData, contrast_column: str, contrasts: list[Tuple[str, str]]) -> (pd.DataFrame, pd.DataFrame):
    rnk_stats, num_cells = [], []
    precomputed_stats = {}

    conditions = []
    for condition in contrasts:
        conditions.extend([condition[0], condition[1]])
    conditions = np.unique(conditions)
    
    for condition in conditions:
        precomputed_stats[condition] = compute_mu_and_sigma(
            adata, 
            contrast_column=contrast_column, 
            condition=condition
        )

    for condition in contrasts:
        _rnk_stats, _num_cells = compute_s2n(
            adata,
            contrast_column=contrast_column,
            condition_1=condition[0],
            condition_2=condition[1],
            precomputed_stats=precomputed_stats,
        )
        rnk_stats.append(_rnk_stats)
        num_cells.append(_num_cells)
    return pd.concat(rnk_stats, axis=1), pd.concat(num_cells, axis=0)

    
def run_enrichment_test(
    adata: AnnData,
    df_hcd_all: pd.DataFrame,
    celltype: Tuple[str, str] = ("B cell", "B"),
    direction: str = "upregulated",
    threshold_pval: float = 0.01,
    threshold_lfc: float = 1.,
    threshold_expression: float = 0,
    contrast_column: str = "disease_state",
    celltype_column: str = "disease_state",
    contrasts: Tuple[str, str] | list[Tuple[str, str]] = None,
    min_size: int = 10,
    max_size: int = 1000,
    permutation_num: int = 1000,
    weight: float = 1.,
    seed: int = 2025,
    verbose: bool = True,
    threads: int = 6,
) -> pd.DataFrame:

    if not isinstance(contrasts, list):
        assert isinstance(contrasts, Tuple)
        contrasts = [contrasts]
        
    celltype_adata = celltype[0]
    celltype_signature = celltype[1]
    
    # filter for cell type
    vprint("Filter for cell type:", verbose)
    adata = adata[adata.obs[celltype_column] == celltype_adata]
    vprint("Filter for cell type: done.", verbose)
    
    # filter based on gene expression
    vprint("Filter for gene expression:", verbose)
    adata = adata[:, adata.X.mean(axis=0) >= threshold_expression]
    vprint("Filter for gene expression: done.", verbose)

    # get genesets
    vprint("Get gene sets:", verbose)
    gene_set_dict, gene_set_df = get_genesets(
        adata=adata,
        df_hcd_all=df_hcd_all,
        celltype_signature=celltype_signature,
        direction=direction,
        threshold_pval=threshold_pval,
        threshold_lfc=threshold_lfc,
    )

    vprint("Get gene sets: done.", verbose)
    
    # compute ranking stat
    vprint("Get ranking stats:", verbose)
    rnk_stats, num_cells_per_condition = compute_ranking_statistic(adata, contrast_column=contrast_column, contrasts=contrasts)
    vprint("Get ranking stats: done.", verbose)
    results = []
    
    for contrast_name in rnk_stats.columns:
        print(contrast_name)
        # format stat so that it can be processed be gseapy
        rnk = rnk_stats.loc[:, contrast_name]\
            .replace([np.inf, -np.inf], np.nan)\
            .dropna()\
            .sort_values(ascending=False)\
            .to_frame()\
            .rename({contrast_name: 1}, axis=1)\
            .rename_axis("0")
        
        # run enrichment
        gp_res = gp.prerank(
            rnk=rnk,
            gene_sets=gene_set_dict,
            min_size=min_size,
            max_size=max_size,
            permutation_num=permutation_num,
            weight=weight,
            outdir=None,
            seed=seed,
            verbose=verbose,
            threads=threads,
        )
        _res = gp_res.res2d
        _res.loc[:, "contrast"] = contrast_name
        _res.loc[:, "num_cells_1"] = num_cells_per_condition.loc[contrast_name, "num_cells_1"]
        _res.loc[:, "num_cells_2"] = num_cells_per_condition.loc[contrast_name, "num_cells_2"]
        _res.loc[:, "percent_duplicate_ranking_stats"] = (rnk.duplicated(keep="first").sum() / rnk.shape[0]) * 100
        results.append(_res)
        vprint(f"{contrast_name}: done.", verbose)

    # combine results and save hyperparams
    results = pd.concat(results, axis=0, ignore_index=True)
    results.loc[:, "celltype_adata"] = celltype_adata
    results.loc[:, "celltype_signature"] = celltype_signature
    results.loc[:, "celltype_combo"] = f"{celltype_adata} ({celltype_signature})"
    results.loc[:, "direction"] = direction
    results.loc[:, "threshold_pval"] = threshold_pval
    results.loc[:, "threshold_lfc"] = threshold_lfc
    results.loc[:, "threshold_expression"] = threshold_expression
    results.loc[:, "min_size"] = min_size
    results.loc[:, "max_size"] = max_size
    results.loc[:, "permutation_num"] = permutation_num
    results.loc[:, "weight"] = weight
    results.loc[:, "seed"] = seed
    results.loc[:, "threads"] = threads

    results.rename({"Term": "cytokine"}, inplace=True, axis=1)
    results = pd.merge(results, gene_set_df, on="cytokine")
    
    return results

In [ ]:
fig_path = "/Users/soeren.becker/repos/HumanCytokineDict/notebooks/08_usecase_MS/figures_manuscript3"
os.makedirs(fig_path, exist_ok=True)

In [ ]:
df_hcd_all = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_all_with_pbs_DE_count.csv", index_col=0)
questionable_genes = set(pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/notebooks/01_overview/questionable_genes.csv")['gene'])
df_hcd_all = df_hcd_all.loc[
    (
        (df_hcd_all['pbs_well'] > 3) 
        & (~df_hcd_all['gene'].isin(questionable_genes))
        & (df_hcd_all['adj_p_value'] < 0.05)
    )
]

In [ ]:
adata = sc.read_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/use_case_datasets/08_Multiple_Sclerose/Schafflick20_MS_CSF.h5ad")
invalid_labels = [
    "Mono Doublet",
    "contamination1",
    "B cell doublets",
    "doublet",
]
adata = adata[~adata.obs.labels.isin(invalid_labels)]

In [ ]:
hcd_var_names = df_hcd_all.gene.unique()
print(len(hcd_var_names))

usecase_var_names = adata.var_names.values
print(len(usecase_var_names))

print(len(np.intersect1d(usecase_var_names, hcd_var_names)))

In [ ]:
np.array(list(adata.obs.labels.unique()))

In [ ]:
df_hcd_all.celltype.unique()

In [ ]:
celltype_combos = ( # cell type in RA dataset, celltype of gene set signature
    ### T cells ###
    ('CD4', "CD4_T_cell"),
    ('CD8a', "CD8_T_cell"),
    ('CD8n', "CD8_T_cell"),
    ('Tregs', "Treg"),
    ### Monocytes ###
    ('Mono', "CD14_Mono"),
    ('ncMono', "CD16_Mono"),
    ### B cells ###
    ('B1', "B_cell"), 
    ('B2', "B_cell"),
    ### Natural killer cells ###
    ('NK1', 'NK_CD56low'),
    ('NK2', 'NK_CD56hi'),
    ### dendritic cells ###
    ('mDC1', "cDC"),
    ('mDC2', "cDC"),
    #('pDC', "pDC"),
    
    # ('plasma', 
    # ('RBC',
    # ('MegaK'
    # ('Gran',
    # ('Tdg', 
)

In [ ]:
# fontsize
fs = 6

In [ ]:
def shorten_ylabels(x):
    if isinstance(x, list):
        return [shorten_ylabels(_x) for _x in x]
    text = x.get_text() if hasattr(x, 'get_text') else x

    text = text.replace("CD4-positive, alpha-beta T cell", r"CD4, $\alpha$-$\beta$ T")
    text = text.replace("CD8-positive, alpha-beta T cell", r"CD8, $\alpha$-$\beta$ T")
    text = text.replace("conventional dendritic cell", "cDC")
    text = text.replace("natural killer cell", "NK")
    text = text.replace("B1", "B cell")
    text = text.replace("Mono", "CD14 Mono.")

    if not text.startswith("NK"):
        text = text.split("(")[0]
    else:
        text = text.replace("CD56-bright_NK", "CD56hi")
        text = text.replace("CD56-dim_NK", "CD56low")
    
    if text.startswith("non-classical monocyte"):
        text =  "nc. Mono."
    elif text.startswith("classical monocyte"):
        text =  "c. Mono."
        
    return text


def shorten_xlabels(x):
    if isinstance(x, list):
        return [shorten_xlabels(_x) for _x in x]

    text = x.get_text() if hasattr(x, 'get_text') else x

    text = text.split("-", 1)[1]
    text = text.replace("beta", r"$\beta$")
    text = text.replace("gamma", r"$\gamma$")
    text = text.replace("alpha", r"$\alpha$")
    text = text.replace("omega", r"$\omega$")
    return text


def shorten_xlabels2(x):
    if isinstance(x, list):
        return [shorten_xlabels2(_x) for _x in x]

    text = x.get_text() if hasattr(x, 'get_text') else x
    print(text)
    text = "-".join(text.split("-", 1)[:-1])
    text = text.replace("beta", r"$\beta$")
    text = text.replace("gamma", r"$\gamma$")
    text = text.replace("alpha", r"$\alpha$")
    text = text.replace("omega", r"$\omega$")
    return text

In [ ]:
adata.obs.labels.value_counts()

### CSF

In [ ]:
result_dir = "./results_final_DEGs"
os.makedirs(result_dir, exist_ok=True)

In [ ]:
threshold_pval = 0.05

all_results_csf = []

for threshold_lfc in [0.8, 0.9, 1, 1.1, 1.2]:
    for threshold_expression in [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]:

        shared_kwargs_csf= {
            "threshold_pval": threshold_pval,
            "threshold_lfc": threshold_lfc,
            "threshold_expression": threshold_expression,
            "direction": "upregulated",
            "min_size": 8,
            "max_size": 1_000,
            "permutation_num": 20_000,
            "verbose": True,
            "adata": adata[adata.obs.CSF == "True"],
            "df_hcd_all": df_hcd_all,
        }
        
        for celltype_combo_k, celltype_combo in enumerate(celltype_combos):
            print(threshold_lfc, threshold_expression, celltype_combo)
            results_csf = run_enrichment_test(
                celltype=celltype_combo,
                contrast_column="MS",
                celltype_column="labels",
                contrasts=[("True", "False")],
                **shared_kwargs_csf,
            )
            
            all_results_csf.append(results_csf)

all_results_csf = pd.concat(all_results_csf, axis=0)
all_results_csf.loc[:, "CSF"] = True
all_results_csf.to_csv(os.path.join(result_dir, "results_MS_csf_robustness_new.csv"))

In [ ]:
del all_results_csf

### Blood

In [ ]:
threshold_pval = 0.05

all_results_blood = []

for threshold_lfc in [0.8, 0.9, 1, 1.1, 1.2]:
    for threshold_expression in [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]:

        shared_kwargs_csf= {
            "threshold_pval": threshold_pval,
            "threshold_lfc": threshold_lfc,
            "threshold_expression": threshold_expression,
            "direction": "upregulated",
            "min_size": 8,
            "max_size": 1_000,
            "permutation_num": 20_000,
            "verbose": True,
            "adata": adata[adata.obs.CSF == "False"],
            "df_hcd_all": df_hcd_all,
        }
        
        for celltype_combo_k, celltype_combo in enumerate(celltype_combos):
        
            results_blood = run_enrichment_test(
                celltype=celltype_combo,
                contrast_column="MS",
                celltype_column="labels",
                contrasts=[("True", "False")],
                **shared_kwargs_csf,
            )
            
            all_results_blood.append(results_blood)

all_results_blood = pd.concat(all_results_blood, axis=0)
all_results_blood.loc[:, "CSF"] = False
all_results_blood.to_csv(os.path.join(result_dir, "results_MS_blood_robustness_new.csv"))

In [ ]:
from tqdm.auto import tqdm
import warnings

def _check_robustness_fractions(
    df_pivot, 
    threshold_qval=0.1, # adjusted p value
    threshold_valid=0.1, # fraction of results required to even consider this condition. I.e. if the test only ran for one set of thresholds, then it is not very robust.
    threshold_below_alpha=0.75, # fraction of results that need to be significant
):
    n_total = np.prod(df_pivot.shape)
    n_valid = n_total - df_pivot.isna().sum().sum()
    n_below_alpha = (df_pivot < threshold_qval).sum().sum() # number of results below pval threshold, i.e., number of significant results
    frac_valid_results = (n_valid / n_total)
    frac_pval_below_alpha = (n_below_alpha / n_valid) # fraction of significant results relative to valid results
    is_robust = (frac_pval_below_alpha > threshold_below_alpha) & (frac_valid_results > threshold_valid)
    return frac_valid_results, frac_pval_below_alpha, is_robust

def check_robustness(
    all_results,
    THRESHOLD_QVAL = 0.1,
    THRESHOLD_VALID = 0.1,
    THRESHOLD_BELOW_ALPHA = 0.9,
):
    
    all_thresholds_expression = all_results.threshold_expression.sort_values(ascending = False).unique()
    all_thresholds_lfc = sorted(all_results.threshold_lfc.unique())
    
    df = pd.DataFrame(index=all_thresholds_expression, columns=all_thresholds_lfc)
    df.index.rename("threshold_expression", inplace=True)
    df.columns.rename("threshold_lfc", inplace=True)
    
    robust_results = []
    
    for contrast in tqdm(all_results.contrast.unique()):
        for celltype_combo in all_results.celltype_combo.unique():
            results_ct = all_results.loc[(all_results.celltype_combo == celltype_combo) & (all_results.contrast == contrast)]
            for cytokine in results_ct.cytokine.unique():
                results_ct_cy = results_ct.loc[results_ct.cytokine == cytokine]
                df_pivot = results_ct_cy.pivot(index="threshold_expression", columns="threshold_lfc", values="FDR q-val")
                row_order = df_pivot.index.sort_values(ascending=False)
                with warnings.catch_warnings():
                    warnings.simplefilter(action='ignore', category=FutureWarning)
                    df_combined = pd.concat([df, df_pivot])
                df_merged = df_combined.combine_first(df_pivot)
                df_merged = df_merged.loc[~df_merged.index.duplicated()]
                df_pivot = df_merged.loc[all_thresholds_expression, all_thresholds_lfc].astype(float)
                frac_valid_results, frac_pval_below_alpha, is_robust = _check_robustness_fractions(
                    df_pivot, 
                    threshold_qval=THRESHOLD_QVAL, 
                    threshold_valid=THRESHOLD_VALID,
                    threshold_below_alpha=THRESHOLD_BELOW_ALPHA
                )
                
                if is_robust:
                    robust_results.append(
                        (
                            celltype_combo, contrast, cytokine,
                            frac_valid_results, frac_pval_below_alpha, is_robust,
                            results_ct_cy.NES.min(), results_ct_cy.NES.max(),
                            THRESHOLD_QVAL,
                            THRESHOLD_BELOW_ALPHA,
                        )
                    )
        
    robust_results = pd.DataFrame(robust_results).rename(
        {0: "celltype_combo", 1: "contrast", 2: "cytokine", 3: "frac_valid", 4: "frac_significant", 5:"is_robust", 6:"NES_min", 7:"NES_max", 8:"qval_threshold", 9: "threshold_frac_below_alpha"},
        axis=1
    )
    return robust_results


program_colors = {
    'CE': '#8dd3c7',
    'CS': '#ffffb3',
    'ES': '#bebada',
    'FC': '#fb8072',
    'GR': '#80b1d3',
    'II': '#fdb462',
    'IT': '#b3de69',
    'MD': '#fccde5',
    'PD': '#d9d9d9',
    'SN': '#bc80bd',
    'TR': '#ccebc5'
}

def clean_celltype_combo_labels(x):
    if isinstance(x, list):
        return [clean_celltype_combo_labels(xx) for xx in x]
    return x.split(" (")[0]

def find_switch_indices(lst):
    """Return indices where the value changes from the previous one."""
    return [0] + [i for i in range(1, len(lst)) if lst[i] != lst[i - 1]]

In [ ]:
all_results_blood = pd.read_csv("./results_final_DEGs/results_MS_blood_robustness_new.csv", index_col=0)
all_results_csf = pd.read_csv("./results_final_DEGs/results_MS_csf_robustness_new.csv", index_col=0)

In [ ]:
THRESHOLD_BELOW_ALPHA = 2/3
THRESHOLD_VALID = 1/3

robust_results_blood_01 = check_robustness(all_results_blood, THRESHOLD_QVAL=0.1, THRESHOLD_VALID = THRESHOLD_VALID, THRESHOLD_BELOW_ALPHA=THRESHOLD_BELOW_ALPHA)
robust_results_blood_005 = check_robustness(all_results_blood, THRESHOLD_QVAL=0.05, THRESHOLD_VALID = THRESHOLD_VALID, THRESHOLD_BELOW_ALPHA=THRESHOLD_BELOW_ALPHA)
robust_results_blood_001 = check_robustness(all_results_blood, THRESHOLD_QVAL=0.01, THRESHOLD_VALID = THRESHOLD_VALID, THRESHOLD_BELOW_ALPHA=THRESHOLD_BELOW_ALPHA)
robust_results_blood = pd.concat([robust_results_blood_01, robust_results_blood_005, robust_results_blood_001])
robust_results_blood = robust_results_blood.groupby(["celltype_combo", "contrast", "cytokine"])["qval_threshold"].min().to_frame().reset_index()

In [ ]:
robust_results_csf_01 = check_robustness(all_results_csf, THRESHOLD_QVAL=0.05, THRESHOLD_VALID = THRESHOLD_VALID, THRESHOLD_BELOW_ALPHA=THRESHOLD_BELOW_ALPHA)
robust_results_csf_005 = check_robustness(all_results_csf, THRESHOLD_QVAL=0.05, THRESHOLD_VALID = THRESHOLD_VALID, THRESHOLD_BELOW_ALPHA=THRESHOLD_BELOW_ALPHA)
robust_results_csf_001 = check_robustness(all_results_csf, THRESHOLD_QVAL=0.01, THRESHOLD_VALID = THRESHOLD_VALID, THRESHOLD_BELOW_ALPHA=THRESHOLD_BELOW_ALPHA)
robust_results_csf = pd.concat([robust_results_csf_01, robust_results_csf_005, robust_results_csf_001])
robust_results_csf = robust_results_csf.groupby(["celltype_combo", "contrast", "cytokine"])["qval_threshold"].min().to_frame().reset_index()

In [ ]:
idx1 = robust_results_blood.set_index(["celltype_combo", "cytokine"]).index
idx2 = robust_results_csf.set_index(["celltype_combo", "cytokine"]).index
idx_all = idx1.union(idx2)

In [ ]:
results_blood = all_results_blood.groupby(["celltype_combo", "cytokine"])["NES"].mean().reset_index()
results_blood = results_blood.set_index(["celltype_combo", "cytokine"])
results_blood = results_blood.loc[results_blood.index.intersection(idx_all)].reset_index()
results_blood.loc[:, "CSF"] = False

In [ ]:
results_csf = all_results_csf.groupby(["celltype_combo", "cytokine"])["NES"].mean().reset_index()
results_csf = results_csf.set_index(["celltype_combo", "cytokine"])
results_csf = results_csf.loc[results_csf.index.intersection(idx_all)].reset_index()
results_csf.loc[:, "CSF"] = True

In [ ]:
df_both = pd.concat([results_blood, results_csf], axis=0).sort_values(["cytokine", "celltype_combo", "CSF"])
df_both.loc[:, "program_ct"] = df_both.cytokine + "___" + df_both.celltype_combo

In [ ]:
df_both = pd.concat([results_blood, results_csf], axis=0).sort_values(["cytokine", "celltype_combo", "CSF"])
df_both.loc[:, "program_ct"] = df_both.cytokine + "___" + df_both.celltype_combo

df_pivot = df_both.sort_values(["celltype_combo", "cytokine", "CSF"]).pivot(
    index="program_ct", 
    columns="CSF", 
    values="NES",
).astype(float).rename({False: "blood", True: "csf"}, axis=1)

df_annot = df_pivot.copy()
df_annot.loc[:, :] = ""

for row in df_annot.index:
    for col in df_annot.columns:
        program, celltype_combo = row.split("___")
        
        if col == "blood":
            select = ((robust_results_blood.cytokine == program) & (robust_results_blood.celltype_combo == celltype_combo))
            alpha = robust_results_blood.loc[select, "qval_threshold"]
        elif col == "csf":
            select = ((robust_results_csf.cytokine == program) & (robust_results_csf.celltype_combo == celltype_combo))
            alpha = robust_results_csf.loc[select, "qval_threshold"]
        
        if len(alpha) == 0:
            df_annot.loc[row, col] = ""
        else:
            assert len(alpha) == 1
            alpha = alpha.values[0]
            if alpha == 0.1:
                df_annot.loc[row, col] = "*"
            elif alpha == 0.05:
                df_annot.loc[row, col] = "**"
            elif alpha == 0.01:
                df_annot.loc[row, col] = "***"
        

ytick_labels = df_pivot.index.tolist()
left_labels = [label.split('___')[0] for label in ytick_labels]
right_labels = [label.split('___')[1] for label in ytick_labels]

starts = find_switch_indices([l.split(" ")[0] for l in left_labels])
stops =  starts[1:] + [starts[-1]+1]

fig, ax = plt.subplots(figsize=(1.5, 3.5))
heatmap_out = sns.heatmap(
    df_pivot.fillna(0),
    center=0,
    vmin=-df_both.NES.max(),
    vmax=df_both.NES.max(),
    cmap="RdBu_r",
    square=True,
    annot=df_annot,
    annot_kws={"fontsize": 6, "family": "Arial"},
    fmt="",
    ax=ax,
    cbar=False,
    cbar_kws={"shrink": 0.5},
    lw=0.5,
    linecolor="white",
)

plt.ylabel("")
plt.xlabel("")
ax.set_yticks(np.arange(0.5, 0.5+len(left_labels)))
ax.set_yticklabels(left_labels, rotation=0)
ax.tick_params(axis='y', labelright=False, labelleft=True, length=0)

ax.tick_params(axis='both', which='both', length=0)

if False:
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position('top')
    plt.xticks([0.5, 1.5], ["Blood", "CSF"], fontsize=6, family="Arial", rotation=90, ha="center")
    plt.tight_layout()
    fig.canvas.draw()
    bbox = ax.get_position()
    yrange = bbox.y1 - bbox.y0
    
    for k, (start, stop) in enumerate(zip(starts, stops)):
        color = program_colors[left_labels[start].split("-")[0]]
        ax.vlines(x=-0.2, ymin=start+0.06, ymax=stop-0.06, color=color, lw=3)
        x_text = bbox.x0 - 0.05
        offset_1 = bbox.y1 - start * ((bbox.y1 - bbox.y0) / df_pivot.shape[0])
        offset_2 = bbox.y1 - stop * ((bbox.y1 - bbox.y0) / df_pivot.shape[0])
        center = (offset_1 - offset_2) / 2
        
        if True:
            fig.text(
                y=offset_1 - center,
                x=x_text,
                s=left_labels[start],
                ha='right',
                va="center",
                fontsize=6,
                family="Arial",
                rotation=0
            )
    
    plt.xlim([-0.2, df_pivot.shape[1]])    
    ax_pos = ax.get_position()
    fig_width = fig.get_figwidth()
    ax_right = fig.add_axes([
        # ax_pos.x1 + 0.035,  
        ax_pos.x1 + 0.08,  
        ax_pos.y0,
        0.1,
        ax_pos.height,
    ])
    ax_right.set_xticks([])
    ax_right.set_ylim(ax.get_ylim())
    ax_right.set_yticks(ax.get_yticks())
    ax_right.set_yticklabels(
        clean_celltype_combo_labels(right_labels),
        rotation=0,
        ha="left",
        fontsize=6,
        family="Arial",
    )
    ax.set_yticklabels([], rotation=0)
    
    ax_right.tick_params(left=False, right=False)
    for spine in ax_right.spines.values():
        spine.set_visible(False)

if False:
    plt.savefig(
        os.path.join("./results/enrichment_MS_cytokines_final_DEGs___.svg"),
        transparent=True,
        dpi=400,
        bbox_inches="tight",
        pad_inches=0,
    )
plt.show()

In [ ]:
all_results_blood.loc[:, "tissue"] = "blood"
all_results_csf.loc[:, "tissue"] = "csf"
all_results = pd.concat([all_results_blood, all_results_csf], axis=0)

In [ ]:
def clean_cytokine_label(x):
    x = x.replace("alpha", r"$\alpha$")
    x = x.replace("beta", r"$\beta$")
    x = x.replace("gamma", r"$\gamma$")
    x = x.replace("omega", r"$\omega$")
    return x

def clean_celltype_label(x):
    if isinstance(x, list):
        return [clean_celltype_label(_x) for _x in x]
    text = x.get_text() if hasattr(x, 'get_text') else x

    if not text.startswith("NK"):
        text = text.split("(", 1)[0]
    else:
        text = text.replace("CD56-dim_NK", "CD56low")
        text = text.replace("CD56-bright_NK", "CD56high")
    
    return text

In [ ]:
all_celltypes = [
    'B1 (B)', 
    'B2 (B)',
    'CD4 (CD4)', 
    'CD8a (CD8)', 
    'CD8n (CD8)',
    'Tregs (Treg)',
    'NK1 (CD56-bright_NK)',
    'NK1 (CD56-dim_NK)',
    'NK2 (CD56-bright_NK)',
    'NK2 (CD56-dim_NK)',
    'Mono (CD14_Mono)',
    'ncMono (CD16_Mono)',
    'mDC1 (cDC)',
    'mDC2 (cDC)',
    
]

significant_celltypes = [
    # 'B1 (B)',
    # 'B2 (B)',
    'CD4 (CD4)', 
    'CD8a (CD8)', 
    'CD8n (CD8)',
    # 'Tregs (Treg)',
    'NK1 (CD56-bright_NK)',
    'NK1 (CD56-dim_NK)',
    'NK2 (CD56-bright_NK)',
    'NK2 (CD56-dim_NK)',
    # 'Mono (CD14_Mono)',
    # 'ncMono (CD16_Mono)',
    # 'mDC1 (cDC)',
    # 'mDC2 (cDC)',
    
]

valid_cytokines = [
    'IL-32-beta', 
    'IL-2',
    'IL-15',    
    'IFN-beta',
    'IFN-gamma', 
    'IFN-omega',
    'IL-1-beta',
    'IL-4',
    'IL-7',
]

valid_tissues = ["csf", "blood"]

In [ ]:
fig_path = "./figure_manuscript_final"
os.makedirs(fig_path, exist_ok=True)

### VERTICAL: all results

In [ ]:
THRESHOLD_EXPRESSION = 0.05
THRESHOLD_LFC = 1
QVAL_THRESHOLD = 0.05

selected_results = all_results.loc[(
    all_results.celltype_combo.isin(all_celltypes) 
    & all_results.cytokine.isin(valid_cytokines) 
    & (all_results.threshold_expression == THRESHOLD_EXPRESSION)
    & (all_results.threshold_lfc == THRESHOLD_LFC)
)]


df_pivot = selected_results.pivot(
    index=("cytokine", "tissue"),
    columns="celltype_combo",
    values="NES"
).astype(float)
multiindex_order = pd.MultiIndex.from_tuples([(c, cy) for c in valid_cytokines for cy in valid_tissues], names=df_pivot.T.columns.names)
df_pivot = df_pivot.T.reindex(columns=multiindex_order).T

df_pivot_qval = selected_results.pivot(
    index=("cytokine", "tissue"),
    columns="celltype_combo",
    values="FDR q-val"
).astype(float)
df_pivot_qval = df_pivot_qval.T.reindex(columns=multiindex_order).T

annot = df_pivot.copy().astype(str)

for row in df_pivot.index:
    for col in df_pivot.columns:
        val = df_pivot.loc[row, col]
        qval = df_pivot_qval.loc[row, col]
        if pd.notna(val) and pd.notna(qval) and qval < QVAL_THRESHOLD:
            annot.loc[row, col] = f"{val:.1f}"
        else:
            annot.loc[row, col] = ""


fig, ax = plt.subplots(1, 1, figsize=(4., 4.))
sns.heatmap(
    df_pivot,
    square=True,
    annot=annot,
    cmap="RdBu_r",
    center=0,
    annot_kws={"fontsize": 6, "family": "Arial"},
    linecolor="white",
    fmt="",
    linewidths=0.01,
    cbar=False,
    ax=ax,
)
plt.xlabel("")
plt.ylabel("")
plt.yticks([])
ax.tick_params(axis='both', which='both', length=0)

fig.canvas.draw()
bbox = ax.get_position()
yrange = bbox.y1 - bbox.y0

for start, stop in zip(range(df_pivot.shape[0]+1)[:-1], range(df_pivot.shape[0]+1)[1:]):
    if start % 2 == 0:
        color = "black"
    else:
        color = "darkgray"
    ax.vlines(x=-0.2, ymin=start, ymax=stop, color=color)

n_rows = df_pivot.shape[0]
for y_i, (y, cytokine) in enumerate(zip(range(0, n_rows, 2), df_pivot.reset_index().cytokine[::2][::-1])):
    ax.hlines(y, xmin=-0.2, xmax=df_pivot.shape[1], colors='white', linewidth=1.5)
    offset = bbox.y0 + (yrange/(df_pivot.shape[0]))
    center = offset + (y_i * (yrange/(df_pivot.shape[0]/2)))
    x_text = bbox.x0 - 0.05
    cytokine = clean_cytokine_label(cytokine)
    fig.text(y=center, x=x_text, s=cytokine, ha='center', va="center", fontsize=fs, family="Arial", rotation=0)

xtick_positions = plt.gca().get_xticks()
xtick_labels = plt.gca().get_xticklabels()
xtick_labels_clean = clean_celltype_label(xtick_labels)
plt.xticks(xtick_positions, xtick_labels_clean, fontsize=fs, family="Arial", va="top")
    
plt.xlim([-0.2, df_pivot.shape[1]])
plt.savefig(
    os.path.join(fig_path, "enrichment_MS_all_vertical.svg"),
    transparent=True,
    dpi=400,
    bbox_inches="tight",
    pad_inches=0,
)
plt.show()

### VERTICAL: significant results

In [ ]:
THRESHOLD_EXPRESSION = 0.05
THRESHOLD_LFC = 1
QVAL_THRESHOLD = 0.05 # threshold for significance, indicated by annotation in heatmap

selected_results = all_results.loc[(
    all_results.celltype_combo.isin(significant_celltypes) 
    & all_results.cytokine.isin(valid_cytokines) 
    & (all_results.threshold_expression == THRESHOLD_EXPRESSION)
    & (all_results.threshold_lfc == THRESHOLD_LFC)
)]

df_pivot = selected_results.pivot(
    index=("cytokine", "tissue"),
    columns="celltype_combo",
    values="NES"
).astype(float)
multiindex_order = pd.MultiIndex.from_tuples([(c, cy) for c in valid_cytokines for cy in valid_tissues], names=df_pivot.T.columns.names)
df_pivot = df_pivot.T.reindex(columns=multiindex_order).T

df_pivot_qval = selected_results.pivot(
    index=("cytokine", "tissue"),
    columns="celltype_combo",
    values="FDR q-val"
).astype(float)
df_pivot_qval = df_pivot_qval.T.reindex(columns=multiindex_order).T

annot = df_pivot.copy().astype(str)

for row in df_pivot.index:
    for col in df_pivot.columns:
        val = df_pivot.loc[row, col]
        qval = df_pivot_qval.loc[row, col]
        if pd.notna(val) and pd.notna(qval) and qval < QVAL_THRESHOLD:
            annot.loc[row, col] = f"{val:.1f}"
        else:
            annot.loc[row, col] = ""


fig, ax = plt.subplots(1, 1, figsize=(3.6, 3.6))
sns.heatmap(
    df_pivot,
    square=True,
    annot=annot,
    cmap="RdBu_r",
    center=0,
    annot_kws={"fontsize": 6, "family": "Arial"},
    linecolor="white",
    fmt="",
    linewidths=0.25,
    cbar=False,
    ax=ax,
)
plt.xlabel("")
plt.ylabel("")
plt.yticks([])
ax.tick_params(axis='both', which='both', length=0)
fig.canvas.draw()
bbox = ax.get_position()
yrange = bbox.y1 - bbox.y0

for start, stop in zip(range(df_pivot.shape[0]+1)[:-1], range(df_pivot.shape[0]+1)[1:]):
    if start % 2 == 0:
        color = "black"
    else:
        color = "darkgray"
    ax.vlines(x=-0.2, ymin=start, ymax=stop, color=color)

n_rows = df_pivot.shape[0]
for y_i, (y, cytokine) in enumerate(zip(range(0, n_rows, 2), df_pivot.reset_index().cytokine[::2][::-1])):
    ax.hlines(y, xmin=-0.2, xmax=df_pivot.shape[1], colors='white', linewidth=1.5)
    offset = bbox.y0 + (yrange/(df_pivot.shape[0]))
    center = offset + (y_i * (yrange/(df_pivot.shape[0]/2)))
    x_text = bbox.x0 - 0.05
    cytokine = clean_cytokine_label(cytokine)
    fig.text(y=center, x=x_text, s=cytokine, ha='center', va="center", fontsize=fs, family="Arial", rotation=0)

xtick_positions = plt.gca().get_xticks()
xtick_labels = plt.gca().get_xticklabels()
xtick_labels_clean = clean_celltype_label(xtick_labels)
plt.xticks(xtick_positions, xtick_labels_clean, fontsize=fs, family="Arial", va="top")
plt.xlim([-0.2, df_pivot.shape[1]])
plt.savefig(
    os.path.join(fig_path, "enrichment_MS_significant_vertical.svg"),
    transparent=True,
    dpi=400,
    bbox_inches="tight",
    pad_inches=0,
)
plt.show()

### HORIZONAL: significant results

In [ ]:
THRESHOLD_EXPRESSION = 0.05
THRESHOLD_LFC = 1
QVAL_THRESHOLD = 0.05 # threshold for significance, indicated by annotation in heatmap

LINECOLOR = "white"

selected_results = all_results.loc[(
    all_results.celltype_combo.isin(significant_celltypes) 
    & all_results.cytokine.isin(valid_cytokines) 
    & (all_results.threshold_expression == THRESHOLD_EXPRESSION)
    & (all_results.threshold_lfc == THRESHOLD_LFC)
)]

df_pivot = selected_results.pivot(
    index=("cytokine", "tissue"),
    columns="celltype_combo",
    values="NES"
).astype(float)
multiindex_order = pd.MultiIndex.from_tuples([(c, cy) for c in valid_cytokines for cy in valid_tissues], names=df_pivot.T.columns.names)
df_pivot = df_pivot.T.reindex(columns=multiindex_order).loc[significant_celltypes[::-1]]

df_pivot_qval = selected_results.pivot(
    index=("cytokine", "tissue"),
    columns="celltype_combo",
    values="FDR q-val"
).astype(float)
df_pivot_qval = df_pivot_qval.T.reindex(columns=multiindex_order).loc[significant_celltypes[::-1]]

annot = df_pivot.copy().astype(str)

for row in df_pivot.index:
    for col in df_pivot.columns:
        val = df_pivot.loc[row, col]
        qval = df_pivot_qval.loc[row, col]
        if pd.notna(val) and pd.notna(qval) and qval < QVAL_THRESHOLD:
            annot.loc[row, col] = f"{val:.1f}"
        else:
            annot.loc[row, col] = ""

fig, ax = plt.subplots(1, 1, figsize=(3.7, 3.7))
sns.heatmap(
    df_pivot,
    square=True,
    annot=annot,
    cmap="RdBu_r",
    center=0,
    annot_kws={"fontsize": 6, "family": "Arial"},
    linecolor=LINECOLOR,
    fmt="",
    linewidths=0.25,
    cbar=False,
    ax=ax,
)

plt.xlabel("")
plt.ylabel("")
ax.tick_params(axis='both', which='both', length=0)

fig.canvas.draw()
bbox = ax.get_position()
yrange = bbox.x1 - bbox.x0

for start, stop in zip(range(df_pivot.shape[1]+1)[:-1], range(df_pivot.shape[1]+1)[1:]):
    if start % 2 == 0:
        color = "black"
    else:
        color = "darkgray"
    ax.hlines(y=-0.2, xmin=start, xmax=stop, color=color, linewidth=1)

plt.ylim([-1, df_pivot.shape[0]])
plt.xticks([])

n_cols = df_pivot.shape[1]
for y_i, (x, cytokine) in enumerate(zip(range(0, n_cols, 2), df_pivot.T.reset_index().cytokine[::2])):
    ax.vlines(x, ymin = -0., ymax=df_pivot.shape[1], colors=LINECOLOR, linewidth=1.5)
    offset = bbox.x0  + (yrange/(df_pivot.shape[1]))
    center = offset + (y_i * (yrange/(df_pivot.shape[1]/2)))
    y_text = bbox.y0 - 0.02
    cytokine = clean_cytokine_label(cytokine)
    fig.text(y=y_text , x=center, s=cytokine, ha='center', va="center", fontsize=fs, family="Arial", rotation=0)

ytick_positions = plt.gca().get_yticks()
ytick_labels = plt.gca().get_yticklabels()
ytick_labels_clean = clean_celltype_label(ytick_labels)
plt.yticks(ytick_positions, ytick_labels_clean, fontsize=fs, family="Arial", va="center")

if True:
    plt.savefig(
        os.path.join(fig_path, "enrichment_MS_significant_horizontal.svg"),
        transparent=True,
        dpi=400,
        bbox_inches="tight",
        pad_inches=0,
        )
plt.show()

In [ ]:
cytokine_info = pd.read_excel("/Users/soeren.becker/repos/HumanCytokineDict/annotations/20250125_cytokine_info_with_functional_classification_LV.xlsx")
cytokine_info.head(3)

### cell-cell communication, blood

In [ ]:
adata_blood = adata[adata.obs.CSF == "False"]
print(len(valid_cytokines), cytokine_info.name.isin(valid_cytokines).sum())
cyto_genes = cytokine_info.loc[cytokine_info.name.isin(valid_cytokines)].gene

included_cyto_genes = cyto_genes[np.isin(cyto_genes, adata_blood.var_names)]
missing_cyto_genes = cyto_genes[~np.isin(cyto_genes, adata_blood.var_names)]
display(included_cyto_genes)
display(missing_cyto_genes)
display(adata_blood.var_names[adata_blood.var_names.str.startswith("IFN")])

In [ ]:
all_dfs = []

for celltype in adata_blood.obs.labels.unique():
    _df = pd.DataFrame(
        columns=included_cyto_genes,
        data=np.asarray(adata_blood[adata_blood.obs.labels == celltype, included_cyto_genes].X.mean(axis=0)),
    )
    _df.loc[0, "celltype"] = celltype
    all_dfs.append(_df)

df_expr = pd.concat(all_dfs, axis=0).set_index("celltype")
sns.heatmap(
    df_expr[included_cyto_genes],
    annot=True,
    vmin=0, 
    vmax=0.1,
)

plt.show()

In [ ]:
all_dfs = []

for celltype in adata_blood.obs.labels.unique():
    _df = pd.DataFrame(
        columns=included_cyto_genes,
        data=np.asarray((adata_blood[adata_blood.obs.labels == celltype, included_cyto_genes].X > 0).mean(axis=0)),
    )
    _df.loc[0, "celltype"] = celltype
    all_dfs.append(_df)

df_expr = pd.concat(all_dfs, axis=0).set_index("celltype")
sns.heatmap(
    df_expr[included_cyto_genes],
    annot=True,
    vmin=0, 
    vmax=0.1,
)

plt.show()

### cell-cell communication, csf

In [ ]:
adata_csf = adata[adata.obs.CSF == "False"]
print(len(valid_cytokines), cytokine_info.name.isin(valid_cytokines).sum())
cyto_genes = cytokine_info.loc[cytokine_info.name.isin(valid_cytokines)].gene

included_cyto_genes = cyto_genes[np.isin(cyto_genes, adata_blood.var_names)]
missing_cyto_genes = cyto_genes[~np.isin(cyto_genes, adata_blood.var_names)]
display(included_cyto_genes)
display(missing_cyto_genes)
display(adata_blood.var_names[adata_blood.var_names.str.startswith("IFN")])

In [ ]:
valid_cytokines

In [ ]:
all_dfs = []

for celltype in adata.obs.labels.unique():
    _df = pd.DataFrame(
        columns=included_cyto_genes,
        data=np.asarray((adata[adata.obs.labels == celltype, included_cyto_genes].X > 0).mean(axis=0)),
    )
    _df.loc[0, "celltype"] = celltype
    all_dfs.append(_df)

df_expr = pd.concat(all_dfs, axis=0).set_index("celltype")
sns.heatmap(
    df_expr[included_cyto_genes],
    annot=True,
    vmin=0, 
    vmax=0.1,
    fmt=".2f",
)

plt.show()

In [ ]:
all_dfs = []

for celltype in adata_csf.obs.labels.unique():
    _df = pd.DataFrame(
        columns=included_cyto_genes,
        data=np.asarray((adata_csf[adata_csf.obs.labels == celltype, included_cyto_genes].X > 0).mean(axis=0)),
    )
    _df.loc[0, "celltype"] = celltype
    all_dfs.append(_df)

df_expr = pd.concat(all_dfs, axis=0).set_index("celltype")
sns.heatmap(
    df_expr[included_cyto_genes],
    annot=True,
    vmin=0, 
    vmax=0.01,
    cmap="Reds"
)

plt.show()

In [ ]:
Cytokine	Producing Immune Cell Types
IL-32β	CD4⁺ T cells, CD8⁺ T cells, NK cells, monocytes/macrophages, dendritic cells
IL-2	CD4⁺ T cells (Th1, Th2), CD8⁺ T cells, to a lesser extent NK cells
IL-15	Monocytes, macrophages, dendritic cells, stromal cells, some epithelial cells
IFN-β	Plasmacytoid DCs, monocytes/macrophages, fibroblasts (not typically lymphocytes)
IFN-γ	CD4⁺ Th1 cells, CD8⁺ cytotoxic T cells, NK cells, NKT cells
IFN-ω	Plasmacytoid dendritic cells (pDCs), epithelial cells (low-level in immune cells)
IL-1β	Monocytes, macrophages, dendritic cells, neutrophils
IL-4	CD4⁺ Th2 cells, basophils, eosinophils, mast cells
IL-7	Mainly stromal cells, but also dendritic cells and macrophages under some conditions

### previously significant results

In [ ]:
tissue = "blood"
celltype_combo = "CD4 (CD4)"
cytokine = "IL-7"

fig, ax = plt.subplots(1,1,figsize=(5, 5))
sns.heatmap(
    all_results.loc[(
        (all_results.celltype_combo == celltype_combo)
        &(all_results.cytokine == cytokine)
        &(all_results.tissue == tissue)
    )].pivot(index="threshold_lfc", columns="threshold_expression", values="FDR q-val").astype(float),
    annot=True,
    fmt=".2f",
    square=True,
    ax=ax,
    cbar=False,
    vmin=0,
    vmax=0.1
)
plt.title(f"{tissue} | {celltype_combo} | {cytokine}")
plt.show()

tissue = "csf"

fig, ax = plt.subplots(1,1,figsize=(5, 5))
sns.heatmap(
    all_results.loc[(
        (all_results.celltype_combo == celltype_combo)
        &(all_results.cytokine == cytokine)
        &(all_results.tissue == tissue)
    )].pivot(index="threshold_lfc", columns="threshold_expression", values="FDR q-val").astype(float),
    annot=True,
    fmt=".2f",
    square=True,
    ax=ax,
    cbar=False,
    vmin=0,
    vmax=0.1
)
plt.title(f"{tissue} | {celltype_combo} | {cytokine}")
plt.show()

In [ ]:
tissue = "blood"
celltype_combo = "CD4 (CD4)"
cytokine = "IL-1-beta"

fig, ax = plt.subplots(1,1,figsize=(5, 5))
sns.heatmap(
    all_results.loc[(
        (all_results.celltype_combo == celltype_combo)
        &(all_results.cytokine == cytokine)
        &(all_results.tissue == tissue)
    )].pivot(index="threshold_lfc", columns="threshold_expression", values="FDR q-val").astype(float),
    annot=True,
    fmt=".2f",
    square=True,
    ax=ax,
    cbar=False,
    vmin=0,
    vmax=0.1
)
plt.title(f"{tissue} | {celltype_combo} | {cytokine}")
plt.show()

tissue = "csf"

fig, ax = plt.subplots(1,1,figsize=(5, 5))
sns.heatmap(
    all_results.loc[(
        (all_results.celltype_combo == celltype_combo)
        &(all_results.cytokine == cytokine)
        &(all_results.tissue == tissue)
    )].pivot(index="threshold_lfc", columns="threshold_expression", values="FDR q-val").astype(float),
    annot=True,
    fmt=".2f",
    square=True,
    ax=ax,
    cbar=False,
    vmin=0,
    vmax=0.1
)
plt.title(f"{tissue} | {celltype_combo} | {cytokine}")
plt.show()

### old

In [ ]:
contrast_order = ['True_vs_False']

celltype_order = [ # cell type in RA dataset, celltype of gene set signature
    ### Monocytes ###
    #'Mono (CD14_Mono)',
    #'ncMono (CD16_Mono)',
    ### dendritic cells ###
    #'mDC1 (cDC)',
    'mDC2 (cDC)',
    # ('pDC', "pDC"),
    ### T cells ###
    'CD4 (CD4)',
    'CD8a (CD8)',
    'CD8n (CD8)',
    #'Tregs (Treg)',
    ### Natural killer cells ### 
    #'NK1 (CD56-dim_NK)',
    'NK2 (CD56-dim_NK)',
    #'NK1 (CD56-bright_NK)',
    'NK2 (CD56-bright_NK)',
    ### B cells ###
    #'B1 (B)', 
    #'B2 (B)',

    # ('plasma', 
    # ('RBC',
    # ('MegaK'
    # ('Gran',
    # ('Tdg', 
]

cytokine_order = [
    "IL-1-alpha", "IL-1-beta", "IL-2", "IL-7", "IL-15", "GM-CSF", "IL-12",
    "IFN-alpha1", "IFN-beta", "IFN-omega", "IFN-gamma", "IL-27", "OSM",
    "IL-17C", "TNF-alpha", "IGF-1", "IL-17A", "IL-17D", "IFN-lambda1",
    "CD27L", "RANKL", "G-CSF", "EGF", "TSLP", "IFN-lambda3", "BAFF",
    "TL1A", "FGF-beta", "Noggin", "IL-4", "IL-13", "IL-1Ra", "IL-33",
    "IL-21", "IL-3", "IL-10", "IFN-epsilon", "CD40L", "C5a", "IL-34",
    "M-CSF", "Leptin", "IL-32-beta"
]

In [ ]:
all_results_csf = pd.read_csv(f"./results_MS_csf_robustness.csv", index_col=0)
all_results_csf

In [ ]:
sort_order = ["celltype_combo", "cytokine", "threshold_lfc", "threshold_expression"]
all_results_csf = all_results_csf.sort_values(sort_order)
all_results_csf.loc[:, "percent_duplicate_ranking_stats"] = all_results_csf_stats.sort_values(sort_order).loc[:, "percent_duplicate_ranking_stats"].values



In [ ]:
all_results_csf.columns

In [ ]:
df = (all_results_csf.groupby(["cytokine"])["FDR q-val"].min() < 0.1).to_frame().reset_index()
valid_cytokines = df.loc[df["FDR q-val"]].cytokine.values
print(valid_cytokines)

df = (all_results_csf.groupby(["celltype_combo"])["FDR q-val"].min() < 0.1).to_frame().reset_index()
valid_celltypes = df.loc[df["FDR q-val"]].celltype_combo.values
print(valid_celltypes)

In [ ]:
all_results_csf.contrast.unique()

In [ ]:
all_results_csf.cytokine.unique()

In [ ]:
celltype_order

In [ ]:
all_results_csf = all_results_csf.loc[all_results_csf.celltype_combo.isin(valid_celltypes) & all_results_csf.cytokine.isin(valid_cytokines)]

df_pivot = all_results_csf.pivot(index=["contrast", "cytokine"], columns="celltype_combo", values="NES").loc[:, celltype_order].T
multiindex_order = pd.MultiIndex.from_tuples([(c, cy) for c in contrast_order for cy in cytokine_order], names=df_pivot.columns.names)
df_pivot = df_pivot.reindex(columns=multiindex_order)

df_pivot_qval = all_results_csf.pivot(index=["contrast", "cytokine"], columns="celltype_combo", values="FDR q-val").loc[:, celltype_order].T
df_pivot_qval = df_pivot_qval.reindex(columns=multiindex_order)

### vertical map

In [ ]:
QVAL_THRESHOLD = 0.1

df_pivot_v = df_pivot.T.apply(pd.to_numeric, errors='coerce')
df_pivot_qval_v = df_pivot_qval.T.apply(pd.to_numeric, errors='coerce')

annot_v = df_pivot_v.copy().astype(str)

for row in df_pivot_v.index:
    for col in df_pivot_v.columns:
        val = df_pivot_v.loc[row, col]
        qval = df_pivot_qval_v.loc[row, col]
        if pd.notna(val) and pd.notna(qval) and qval < QVAL_THRESHOLD:
            annot_v.loc[row, col] = f"{val:.1f}"
        else:
            annot_v.loc[row, col] = ""

df_pivot_v_clean_csf = df_pivot_v.dropna(axis=0, how='all').dropna(axis=1, how='all')
annot_v_clean_csf = annot_v.loc[df_pivot_v_clean_csf.index, df_pivot_v_clean_csf.columns]
df_pivot_qval_v_csf = df_pivot_qval_v.loc[df_pivot_v_clean_csf.index, df_pivot_v_clean_csf.columns]

df_pivot_v_clean_csf.index = df_pivot_v_clean_csf.index.set_levels(
    ["CSF" if x == "True_vs_False" else x for x in df_pivot_v_clean_csf.index.levels[0]], level="contrast"
)
df_pivot_qval_v_csf.index = df_pivot_qval_v_csf.index.set_levels(
    ["CSF" if x == "True_vs_False" else x for x in df_pivot_qval_v_csf.index.levels[0]], level="contrast"
)
annot_v_clean_csf.index = annot_v_clean_csf.index.set_levels(
    ["CSF" if x == "True_vs_False" else x for x in df_pivot_v_clean_csf.index.levels[0]], level="contrast"
)

contrast_order = ["CSF"]

fig, ax = plt.subplots(1, 1, figsize=(4., 4.))
sns.heatmap(
    df_pivot_v_clean_csf,
    square=True,
    annot=annot_v_clean_csf,
    cmap="RdBu_r",
    center=0,
    annot_kws={"fontsize": 6, "family": "Arial"},
    linecolor="white",
    fmt="",
    linewidths=0.25,
    cbar=False,
    ax=ax,
)

if True:
    gap_positions = np.cumsum(multiindex_order.to_frame().contrast.value_counts().values)
    for pos in gap_positions:
        ax.vlines(pos, *ax.get_ylim(), colors='white', linewidth=2.)
    
    plt.xlabel("")
    plt.ylabel("")
    ax.tick_params(axis='both', which='both', length=0)
    ytick_positions = plt.gca().get_yticks()
    ytick_labels = plt.gca().get_yticklabels()
    ytick_labels_reduced = shorten_xlabels(ytick_labels)
    plt.yticks(ytick_positions, ytick_labels_reduced, fontsize=fs, family="Arial", ha="right")
    
    xtick_positions = plt.gca().get_xticks()
    xtick_labels = plt.gca().get_xticklabels()
    xtick_labels_reduced = shorten_ylabels(xtick_labels)
    plt.xticks(xtick_positions, xtick_labels_reduced, fontsize=fs, family="Arial", va="top")
    
    fig.canvas.draw()
    bbox = ax.get_position()
    x_line = bbox.y0 - 0.14
    gap_positions = np.array([0] + gap_positions.tolist())
    rel_gap_positions = ((gap_positions / gap_positions[-1])) * (bbox.y1 - bbox.y0)
    offset = bbox.y0
    gap=0.002
    for k, (start, stop, label) in enumerate(zip(rel_gap_positions[0:-1], rel_gap_positions[1:], contrast_order[::-1])):
        if k % 2 == 0:
            linecolor = "black"
        else:
            linecolor = "darkgray"
        linestart = offset + start + gap
        linestop = offset + stop - gap
        
        line = plt.Line2D([x_line, x_line], [linestart, linestop], transform=fig.transFigure, color=linecolor, linewidth=1)
        fig.lines.append(line)
        y_center = linestart + (linestop - linestart)/2
        x_text = 0.
        label = label.replace("_vs_na", "")
        fig.text(y=y_center, x=x_text, s=label, ha='center', va="center", fontsize=fs, family="Arial", rotation=90)
    
    if False:
        plt.savefig(
            os.path.join(fig_path, "enrichment_heatmap_MS_csf_cytokine_vertical.svg"), 
            bbox_inches="tight", 
            pad_inches=0, 
            dpi=500,
            transparent=True,
        )
    plt.show()

# blood

In [ ]:
shared_kwargs_cns = {
    "threshold_pval": 0.01,
    "threshold_lfc": 1,
    "threshold_expression": 0.05,
    "direction": "upregulated",
    "min_size": 8,
    "max_size": 1_000,
    "permutation_num": 20_000,
    "verbose": True,
    "adata": adata[adata.obs.CSF == "False"],
    "df_hcd_all": df_hcd_all,
}

all_results_blood = []

for celltype_combo_k, celltype_combo in enumerate(celltype_combos):

    results_blood = run_enrichment_test(
        celltype=celltype_combo,
        contrast_column="MS",
        celltype_column="labels",
        contrasts=[("True", "False")],
        **shared_kwargs_cns,
    )
    
    all_results_blood.append(results_blood)

all_results_blood = pd.concat(all_results_blood, axis=0)
all_results_blood.loc[:, "CSF"] = False

In [ ]:
all_results_blood.to_csv("./results_MS_blood.csv")

In [ ]:
contrast_order = ['True_vs_False']

celltype_order = [ # cell type in RA dataset, celltype of gene set signature
    ### Monocytes ###
    'Mono (CD14_Mono)',
    #'ncMono (CD16_Mono)',
    ### dendritic cells ###
    #'mDC1 (cDC)',
    'mDC2 (cDC)',
    # ('pDC', "pDC"),
    ### T cells ###
    'CD4 (CD4)',
    'CD8a (CD8)',
    'CD8n (CD8)',
    #'Tregs (Treg)',
    ### Natural killer cells ### 
    'NK1 (CD56-dim_NK)',
    'NK2 (CD56-dim_NK)',
    #'NK1 (CD56-bright_NK)',
    'NK2 (CD56-bright_NK)',
    ### B cells ###
    'B1 (B)', 
    #'B2 (B)',

    # ('plasma', 
    # ('RBC',
    # ('MegaK'
    # ('Gran',
    # ('Tdg', 
]

cytokine_order = [
    "IL-1-alpha", "IL-1-beta", "IL-2", "IL-7", "IL-15", "GM-CSF", "IL-12",
    "IFN-alpha1", "IFN-beta", "IFN-omega", "IFN-gamma", "IL-27", "OSM",
    "IL-17C", "TNF-alpha", "IGF-1", "IL-17A", "IL-17D", "IFN-lambda1",
    "CD27L", "RANKL", "G-CSF", "EGF", "TSLP", "IFN-lambda3", "BAFF",
    "TL1A", "FGF-beta", "Noggin", "IL-4", "IL-13", "IL-1Ra", "IL-33",
    "IL-21", "IL-3", "IL-10", "IFN-epsilon", "CD40L", "C5a", "IL-34",
    "M-CSF", "Leptin", "IL-32-beta"
]

In [ ]:
df = (all_results_blood.groupby(["cytokine"])["FDR q-val"].min() < 0.1).to_frame().reset_index()
valid_cytokines = df.loc[df["FDR q-val"]].cytokine.values
print(valid_cytokines)

df = (all_results_blood.groupby(["celltype_combo"])["FDR q-val"].min() < 0.1).to_frame().reset_index()
valid_celltypes = df.loc[df["FDR q-val"]].celltype_combo.values
print(valid_celltypes)

In [ ]:
all_results_blood = all_results_blood.loc[all_results_blood.celltype_combo.isin(valid_celltypes) & all_results_blood.cytokine.isin(valid_cytokines)]

df_pivot = all_results_blood.pivot(index=["contrast", "cytokine"], columns="celltype_combo", values="NES").loc[:, celltype_order].T
multiindex_order = pd.MultiIndex.from_tuples([(c, cy) for c in contrast_order for cy in cytokine_order], names=df_pivot.columns.names)
df_pivot = df_pivot.reindex(columns=multiindex_order)

df_pivot_qval = all_results_blood.pivot(index=["contrast", "cytokine"], columns="celltype_combo", values="FDR q-val").loc[:, celltype_order].T
df_pivot_qval = df_pivot_qval.reindex(columns=multiindex_order)

#### vertical map

In [ ]:
df_pivot_v = df_pivot.T.apply(pd.to_numeric, errors='coerce')
df_pivot_qval_v = df_pivot_qval.T.apply(pd.to_numeric, errors='coerce')

annot_v = df_pivot_v.copy().astype(str)

for row in df_pivot_v.index:
    for col in df_pivot_v.columns:
        val = df_pivot_v.loc[row, col]
        qval = df_pivot_qval_v.loc[row, col]
        if pd.notna(val) and pd.notna(qval) and qval < QVAL_THRESHOLD:
            annot_v.loc[row, col] = f"{val:.1f}"
        else:
            annot_v.loc[row, col] = ""


df_pivot_v_clean_blood = df_pivot_v.dropna(axis=0, how='all').dropna(axis=1, how='all')
annot_v_clean_blood = annot_v.loc[df_pivot_v_clean_blood.index, df_pivot_v_clean_blood.columns]
df_pivot_qval_v_blood = df_pivot_qval_v.loc[df_pivot_v_clean_blood.index, df_pivot_v_clean_blood.columns]

df_pivot_v_clean_blood.index = df_pivot_v_clean_blood.index.set_levels(
    ["Blood" if x == "True_vs_False" else x for x in df_pivot_v_clean_blood.index.levels[0]], level="contrast"
)

df_pivot_qval_v_blood.index = df_pivot_qval_v_blood.index.set_levels(
    ["Blood" if x == "True_vs_False" else x for x in df_pivot_qval_v_blood.index.levels[0]], level="contrast"
)

annot_v_clean_blood.index = annot_v_clean_blood.index.set_levels(
    ["Blood" if x == "True_vs_False" else x for x in annot_v_clean_blood.index.levels[0]], level="contrast"
)

contrast_order = ["Blood"]

fig, ax = plt.subplots(1, 1, figsize=(4., 4.))
sns.heatmap(
    data=df_pivot_v_clean_blood,
    square=True,
    annot=annot_v_clean_blood,
    cmap="RdBu_r",
    center=0,
    annot_kws={"fontsize": 6, "family": "Arial"},
    linecolor="white",
    fmt="",
    linewidths=0.25,
    cbar=False,
    ax=ax,
)

if True:
    gap_positions = np.cumsum(multiindex_order.to_frame().contrast.value_counts().values)
    for pos in gap_positions:
        ax.vlines(pos, *ax.get_ylim(), colors='white', linewidth=2.)
    
    plt.xlabel("")
    plt.ylabel("")
    ax.tick_params(axis='both', which='both', length=0)
    ytick_positions = plt.gca().get_yticks()
    ytick_labels = plt.gca().get_yticklabels()
    ytick_labels_reduced = shorten_xlabels(ytick_labels)
    plt.yticks(ytick_positions, ytick_labels_reduced, fontsize=fs, family="Arial", ha="right")
    
    xtick_positions = plt.gca().get_xticks()
    xtick_labels = plt.gca().get_xticklabels()
    xtick_labels_reduced = shorten_ylabels(xtick_labels)
    plt.xticks(xtick_positions, xtick_labels_reduced, fontsize=fs, family="Arial", va="top")
    
    fig.canvas.draw()
    bbox = ax.get_position()
    x_line = bbox.y0 - 0.04
    gap_positions = np.array([0] + gap_positions.tolist())
    rel_gap_positions = ((gap_positions / gap_positions[-1])) * (bbox.y1 - bbox.y0)
    offset = bbox.y0
    gap=0.002
    for k, (start, stop, label) in enumerate(zip(rel_gap_positions[0:-1], rel_gap_positions[1:], contrast_order[::-1])):
        if k % 2 == 0:
            linecolor = "black"
        else:
            linecolor = "darkgray"
        linestart = offset + start + gap
        linestop = offset + stop - gap
        # line = plt.Line2D([linestart, linestop], [y_line, y_line], transform=fig.transFigure, color=linecolor, linewidth=1)
        line = plt.Line2D([x_line, x_line], [linestart, linestop], transform=fig.transFigure, color=linecolor, linewidth=1)
        fig.lines.append(line)
        y_center = linestart + (linestop - linestart)/2
        x_text = 0.04
        label = label.replace("_vs_na", "")
        fig.text(y=y_center, x=x_text, s=label, ha='center', va="center", fontsize=fs, family="Arial", rotation=90)
    
    if False:
        plt.savefig(
            os.path.join(fig_path, "enrichment_heatmap_MS_blood_cytokine_vertical.svg"), 
            bbox_inches="tight", 
            pad_inches=0, 
            dpi=500,
            transparent=True,
        )
    plt.show()

In [ ]:
df_pivot_v_clean_blood

In [ ]:
df_pivot_v_clean_csf

### blood and csf

In [ ]:
QVAL_THRESHOLD = 0.1

contrast_order = ["CSF", "Blood"]

df_both = pd.concat([df_pivot_v_clean_blood, df_pivot_v_clean_csf], axis=0, join="outer")
df_qval_both = pd.concat([df_pivot_qval_v_blood, df_pivot_qval_v_csf], axis=0, join="outer")

multiindex_order = df_both.index

annot_v = df_both.copy().astype(str)

for row in df_both.index:
    for col in df_both.columns:
        val = df_both.loc[row, col]
        qval = df_qval_both.loc[row, col]
        if pd.notna(val) and pd.notna(qval) and qval < QVAL_THRESHOLD:
            annot_v.loc[row, col] = f"{val:.1f}"
        else:
            annot_v.loc[row, col] = ""


fig, ax = plt.subplots(1, 1, figsize=(3., 3.))
sns.heatmap(
    data=df_both,
    square=True,
    annot=annot_v,
    cmap="RdBu_r",
    center=0,
    annot_kws={"fontsize": 6, "family": "Arial"},
    linecolor="white",
    fmt="",
    linewidths=0.25,
    cbar=False,
    ax=ax,
)

if True:
    gap_positions = np.cumsum(multiindex_order.to_frame().contrast.value_counts().values)
    #for pos in gap_positions:
    #    ax.vlines(pos, *ax.get_ylim(), colors='white', linewidth=2.)
    
    plt.xlabel("")
    plt.ylabel("")
    ax.tick_params(axis='both', which='both', length=0)
    ytick_positions = plt.gca().get_yticks()
    ytick_labels = plt.gca().get_yticklabels()
    ytick_labels_reduced = shorten_xlabels(ytick_labels)
    plt.yticks(ytick_positions, ytick_labels_reduced, fontsize=fs, family="Arial", ha="right")
    
    xtick_positions = plt.gca().get_xticks()
    xtick_labels = plt.gca().get_xticklabels()
    xtick_labels_reduced = shorten_ylabels(xtick_labels)
    plt.xticks(xtick_positions, xtick_labels_reduced, fontsize=fs, family="Arial", va="top")
    
    fig.canvas.draw()
    bbox = ax.get_position()
    x_line = bbox.y0 + 0.05
    gap_positions = np.array([0] + gap_positions.tolist())
    rel_gap_positions = ((gap_positions / gap_positions[-1])) * (bbox.y1 - bbox.y0)
    offset = bbox.y0
    gap=0.002
    for k, (start, stop, label) in enumerate(zip(rel_gap_positions[0:-1], rel_gap_positions[1:], contrast_order[::-1])):
        if k % 2 == 0:
            linecolor = "black"
        else:
            linecolor = "darkgray"
        linestart = offset + start + gap
        linestop = offset + stop - gap
        # line = plt.Line2D([linestart, linestop], [y_line, y_line], transform=fig.transFigure, color=linecolor, linewidth=1)
        line = plt.Line2D([x_line, x_line], [linestart, linestop], transform=fig.transFigure, color=linecolor, linewidth=1)
        fig.lines.append(line)
        y_center = linestart + (linestop - linestart)/2
        x_text = 0.13
        label = label.replace("_vs_na", "")
        fig.text(y=y_center, x=x_text, s=label, ha='center', va="center", fontsize=fs, family="Arial", rotation=90)
    
    if True:
        plt.savefig(
            os.path.join(fig_path, "enrichment_heatmap_MS_both_vertical.svg"), 
            bbox_inches="tight", 
            pad_inches=0, 
            dpi=500,
            transparent=True,
        )
    plt.show()

In [ ]:
df_both = df_both.swaplevel("contrast", "cytokine").sort_index(level='cytokine')

multiindex_order = df_both.index

fig, ax = plt.subplots(1, 1, figsize=(4., 4.))
sns.heatmap(
    data=df_both,
    square=True,
    annot=True,
    cmap="RdBu_r",
    center=0,
    annot_kws={"fontsize": 6, "family": "Arial"},
    linecolor="white",
    fmt=".1f",
    linewidths=0.25,
    cbar=False,
    ax=ax,
)

plt.xlabel("")
plt.ylabel("")
plt.show()

In [ ]:
df_both

In [ ]:
contrast_order

In [ ]:
results_blood = pd.read_csv("./results_MS_blood.csv")
results_blood.loc[:, "contrast"] = "blood"
results_csf = pd.read_csv("./results_MS_csf.csv")
results_csf.loc[:, "contrast"] = "csf"

In [ ]:
results_blood.loc[results_blood.cytokine == "IL-32-beta"]

In [ ]:
results_csf

In [ ]:
results_both = pd.concat([results_csf, results_blood], axis=0)

In [ ]:
df = (results_both.groupby(["cytokine"])["FDR q-val"].min() < 0.1).to_frame().reset_index()
valid_cytokines = df.loc[df["FDR q-val"]].cytokine.values
print(valid_cytokines)

df = (results_both.groupby(["celltype_combo"])["FDR q-val"].min() < 0.1).to_frame().reset_index()
valid_celltypes = df.loc[df["FDR q-val"]].celltype_combo.values
print(valid_celltypes)

In [ ]:
contrast_order = ["blood", "csf"]
results_both = results_both.loc[results_both.celltype_combo.isin(valid_celltypes) & results_both.cytokine.isin(valid_cytokines)]

In [ ]:
df_pivot = results_both.pivot(index=["contrast", "cytokine"], columns="celltype_combo", values="NES").loc[:, celltype_order].T
multiindex_order = pd.MultiIndex.from_tuples([(c, cy) for c in contrast_order for cy in cytokine_order], names=df_pivot.columns.names)
df_pivot = df_pivot.reindex(columns=multiindex_order)
df_pivot = df_pivot.dropna(axis=0, how='all').dropna(axis=1, how='all')

df_pivot_qval = results_both.pivot(index=["contrast", "cytokine"], columns="celltype_combo", values="FDR q-val").loc[:, celltype_order].T
df_pivot_qval = results_both.reindex(columns=multiindex_order)

In [ ]:
df_pivot_v = df_pivot.T
df_pivot_v = df_pivot_v.swaplevel("contrast", "cytokine").sort_index(level='cytokine')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8., 8.))
sns.heatmap(
    data=df_pivot_v,
    square=True,
    # annot=annot_v_clean_blood,
    annot=True,
    
    cmap="RdBu_r",
    center=0,
    annot_kws={"fontsize": 6, "family": "Arial"},
    linecolor="white",
    fmt=".2f",
    linewidths=0.25,
    cbar=False,
    ax=ax,
)

if True:
    gap_positions = np.cumsum(multiindex_order.to_frame().cytokine.value_counts().values)
    for pos in gap_positions:
        ax.vlines(pos, *ax.get_ylim(), colors='white', linewidth=2.)
    
    plt.xlabel("")
    plt.ylabel("")
    ax.tick_params(axis='both', which='both', length=0)
    ytick_positions = plt.gca().get_yticks()
    ytick_labels = plt.gca().get_yticklabels()
    ytick_labels_reduced = shorten_xlabels2(ytick_labels)
    plt.yticks(ytick_positions, ytick_labels_reduced, fontsize=fs, family="Arial", ha="right")
    
    xtick_positions = plt.gca().get_xticks()
    xtick_labels = plt.gca().get_xticklabels()
    xtick_labels_reduced = shorten_ylabels(xtick_labels)
    plt.xticks(xtick_positions, xtick_labels_reduced, fontsize=fs, family="Arial", va="top")
    
    fig.canvas.draw()
    bbox = ax.get_position()
    x_line = bbox.y0 + 0.07
    gap_positions = np.array([0] + gap_positions.tolist())
    rel_gap_positions = ((gap_positions / gap_positions[-1])) * (bbox.y1 - bbox.y0)
    offset = bbox.y0
    gap=0.002
    for k, (start, stop, label) in enumerate(zip(rel_gap_positions[0:-1], rel_gap_positions[1:], cytokine_order[::-1])):
        if k % 2 == 0:
            linecolor = "black"
        else:
            linecolor = "darkgray"
        linestart = offset + start + gap
        linestop = offset + stop - gap
        # line = plt.Line2D([linestart, linestop], [y_line, y_line], transform=fig.transFigure, color=linecolor, linewidth=1)
        line = plt.Line2D([x_line, x_line], [linestart, linestop], transform=fig.transFigure, color=linecolor, linewidth=1)
        fig.lines.append(line)
        y_center = linestart + (linestop - linestart)/2
        x_text = 0.15
        label = label.replace("_vs_na", "")
        fig.text(y=y_center, x=x_text, s=label, ha='right', va="center", fontsize=fs, family="Arial", rotation=0)
    
    if False:
        plt.savefig(
            os.path.join(fig_path, "enrichment_heatmap_MS_blood_cytokine_vertical.svg"), 
            bbox_inches="tight", 
            pad_inches=0, 
            dpi=500,
            transparent=True,
        )
    plt.show()

